In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("BramVanroy/conll2003")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [ ]:
raw_datasets["train"][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [ ]:
ner_feature = raw_datasets["train"].features["ner_tags"]
ner_feature

List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Preprocess
Do việc tokenize làm cho các từ bị phân mảnh thành các subword do đó các từ sẽ không khớp với mảng ban đầu. Ta phải ánh xạ lại các giá trị cho đúng nhãn và vị trí.

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )

    labels = []
    for i, original_labels in enumerate(examples["ner_tags"]):
        # Lấy danh sách ánh xạ token -> id của từ gốc
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        new_labels = []
        for word_idx in word_ids:
            # 1. Nếu là token đặc biệt ([CLS], [SEP]...), gán nhãn -100 để bỏ qua khi tính loss
            if word_idx is None:
                new_labels.append(-100)

            # 2. Nếu đây là token đầu tiên của một từ mới
            elif word_idx != previous_word_idx:
                new_labels.append(original_labels[word_idx])

            # 3. Nếu đây là các từ con (subwords) thuộc cùng một từ với token trước đó
            else:
                label = original_labels[word_idx]
                # Nếu nhãn gốc là B-XXX (số lẻ), ta đổi nó thành I-XXX (bằng cách cộng 1)
                if label % 2 == 1:
                    label += 1
                new_labels.append(label)
            previous_word_idx = word_idx

        labels.append(new_labels)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
print("Tokens gốc:  ", raw_datasets["train"][0]["tokens"])
print("Nhãn gốc:    ", raw_datasets["train"][0]["ner_tags"])
print("Input IDs:   ", tokenized_datasets["train"][0]["input_ids"])
print("Nhãn mới:    ", tokenized_datasets["train"][0]["labels"])

Tokens gốc:   ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Nhãn gốc:     [3, 0, 7, 0, 0, 0, 7, 0, 0]
Input IDs:    [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102]
Nhãn mới:     [-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]


In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
!pip install evaluate seqeval -q

In [ ]:
!pip install wandb -q

In [ ]:
from huggingface_hub import notebook_login
import wandb

wandb.login()

True

In [ ]:
notebook_login()

In [ ]:
from transformers import AutoModelForTokenClassification

label_list = raw_datasets["train"].features["ner_tags"].feature.names

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [ ]:
import evaluate

metric = evaluate.load("seqeval")

In [ ]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
from transformers import TrainingArguments

model_name = "bert-finetuned-ner-project"
YOUR_HF_USERNAME = "lilith-aeva"

args = TrainingArguments(
    output_dir=model_name,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
    hub_model_id=f"{YOUR_HF_USERNAME}/{model_name}",
    report_to="wandb",
    run_name="ner-experiment-1",
    logging_steps=10,
    fp16=True
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.push_to_hub("Training hoàn tất!")
wandb.finish()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.065489,0.063746,0.897833,0.927297,0.912327,0.981486
2,0.016944,0.054990,0.928347,0.946314,0.937245,0.986092
3,0.022608,0.052151,0.929619,0.949175,0.939296,0.986637


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...project/training_args.bin: 100%|##########| 5.26kB / 5.26kB            

  ...project/model.safetensors:  32%|###1      |  136MB /  431MB            

eval/accuracy,▁▇█
eval/f1,▁▇█
eval/loss,█▃▁
eval/precision,▁██
eval/recall,▁▇█
eval/runtime,▄█▁
eval/samples_per_second,▄▁█
eval/steps_per_second,▄▁█
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇███
train/global_step,▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇███
+3,...
